# Notebook 04 — Math toolkit

**Prerequisites:** complete notebooks **01** through **03** in this curriculum (foundations track). This notebook assumes you are comfortable with `finstack` core types and imports used there.

**In this notebook:** Rust-backed numerics in `finstack.core.math` — linear algebra (Cholesky), descriptive statistics, distribution helpers, and compensated summation — plus a simulation that ties several pieces together.

## Mathematical building blocks

Quant code repeatedly needs a small set of primitives:

- **Symmetric positive-definite (SPD) linear systems** — Cholesky factorization \(A = LL^\top\) and triangular solves appear in calibration, covariance work, and Monte Carlo correlation sampling.
- **Correlation matrices** — must be symmetric, with ones on the diagonal, entries in \([-1,1]\), and positive semi-definite. Invalid matrices break downstream pricing and risk.
- **Robust descriptive statistics** — means, variances, covariances, and quantiles for exploratory checks and validation.
- **Special functions** — Gaussian CDF/PDF and related tools for models and transforms.
- **Stable summation** — naive `sum()` can lose tiny terms when magnitudes differ wildly; compensated methods reduce rounding error.

The APIs below are thin Python bindings over `finstack-core` implementations, so you get consistent numerics across the stack.

### Linear algebra: Cholesky and correlation checks

`cholesky_decomposition` returns a lower-triangular \(L\) with \(A = LL^\top\). `cholesky_solve` solves \(Ax=b\) via **forward** substitution on \(Ly=b\) and **backward** substitution on \(L^\top x=y\) using that Cholesky factor \(L\). `validate_correlation_matrix` in `finstack.core.math.linalg` expects a **nested** square matrix. The same validation is also available for **flat row-major** storage in `finstack.correlation` (handy when matrices are packed in buffers).

In [1]:
from finstack.core.math import linalg
from finstack.core.math.linalg import cholesky_decomposition, cholesky_solve, validate_correlation_matrix
from finstack.correlation import validate_correlation_matrix as validate_correlation_flat

L = cholesky_decomposition([[4.0, 2.0], [2.0, 3.0]])
print("Cholesky L:", L)

x = cholesky_solve(L, [1.0, 1.0])
print("Solution:", x)

validate_correlation_matrix([[1.0, 0.5], [0.5, 1.0]])
print("validate_correlation_matrix (nested 2x2): OK")

validate_correlation_flat([1.0, 0.5, 0.5, 1.0], 2)
print("validate_correlation_matrix (flat row-major, n=2): OK")

print("SINGULAR_THRESHOLD:", linalg.SINGULAR_THRESHOLD)
print("DIAGONAL_TOLERANCE:", linalg.DIAGONAL_TOLERANCE)

Cholesky L: [[2.0, 0.0], [1.0, 1.4142135623730951]]
Solution: [0.125, 0.24999999999999997]
validate_correlation_matrix (nested 2x2): OK
validate_correlation_matrix (flat row-major, n=2): OK
SINGULAR_THRESHOLD: 1e-10
DIAGONAL_TOLERANCE: 1e-06


### Statistics: moments and association

Sample variance uses the \(n-1\) denominator; `population_variance` uses \(n\). `quantile` follows the R-7 / NumPy default with linear interpolation.

In [2]:
from finstack.core.math import stats

data = [1.0, 2.0, 3.0, 4.0, 5.0]
print("Mean:", stats.mean(data))
print("Variance:", stats.variance(data))
print("Pop variance:", stats.population_variance(data))

x = [1.0, 2.0, 3.0, 4.0, 5.0]
y = [2.0, 4.0, 5.0, 4.0, 5.0]
print("Correlation:", stats.correlation(x, y))
print("Covariance:", stats.covariance(x, y))
print("Quantile 50%:", stats.quantile(data, 0.5))

Mean: 3.0
Variance: 2.5
Pop variance: 2.0
Correlation: 0.7745966692414834
Covariance: 1.5
Quantile 50%: 3.0


### Special functions: Gaussian and gamma-related helpers

Standard normal CDF \(\Phi\), PDF \(\varphi\), inverse CDF, `erf`, and \(\ln\Gamma\) for common pricing and statistics workflows.

In [3]:
from finstack.core.math.special_functions import norm_cdf, norm_pdf, standard_normal_inv_cdf, erf, ln_gamma

print("N(0):", norm_cdf(0.0))
print("N(1.96):", norm_cdf(1.96))
print("phi(0):", norm_pdf(0.0))
print("inv_N(0.975):", standard_normal_inv_cdf(0.975))
print("erf(1):", erf(1.0))
print("ln_gamma(5):", ln_gamma(5.0))

N(0): 0.5
N(1.96): 0.9750021048529024
phi(0): 0.3989422804014327
inv_N(0.975): 1.9599639845400538
erf(1): 0.8427007929427149
ln_gamma(5): 3.1780538303479497


### Compensated summation: Kahan and Neumaier

When many small terms combine with large partial sums, floating-point rounding can dominate. **Kahan** is strong when signs are similar; **Neumaier** tends to behave better with mixed signs (e.g., cashflows).

In [4]:
from finstack.core.math.summation import kahan_sum, neumaier_sum

values = [1.0] + [1e-16] * 10000 + [-1.0]
naive = sum(values)
kahan = kahan_sum(values)
neumaier = neumaier_sum(values)
expected = 1e-16 * 10000
print(f"Naive sum:    {naive}")
print(f"Kahan sum:    {kahan}")
print(f"Neumaier sum: {neumaier}")
print(f"Expected:     {expected}")

Naive sum:    9.999999999998685e-13
Kahan sum:    1.000088900582341e-12
Neumaier sum: 9.999999999998685e-13
Expected:     1e-12


## Mini-example: correlated normals and numerical stability

1. Choose a valid \(3 \times 3\) **correlation** matrix \(C\).
2. Factor \(C = LL^\top\) with Cholesky.
3. Draw independent standard normals \(Z\) (`random` module, fixed seed).
4. Set \(X = LZ\); then \(\operatorname{Corr}(X) = C\) (population).
5. Compare **sample** pairwise correlations from many draws to \(C\) using `stats.correlation`.

A second short experiment shows **compensated summation** when a huge positive and negative term nearly cancel: the exact residual is moderate, but a **left-to-right** accumulation (typical `total += x` loops) can lose it to rounding. Note that Python’s built-in `sum()` applies extra care for floats, so we compare against that too.

In [5]:
import random

from finstack.core.math.linalg import cholesky_decomposition, validate_correlation_matrix
from finstack.core.math import stats


def lower_matvec(L, z):
    n = len(z)
    out = [0.0] * n
    for i in range(n):
        acc = 0.0
        for j in range(i + 1):
            acc += L[i][j] * z[j]
        out[i] = acc
    return out


C = [
    [1.0, 0.7, 0.5],
    [0.7, 1.0, 0.3],
    [0.5, 0.3, 1.0],
]
validate_correlation_matrix(C)
print("Target correlation matrix validated: OK")

L = cholesky_decomposition(C)
print("Cholesky L (first row):", L[0])

random.seed(42)
n_samples = 40_000
c0, c1, c2 = [], [], []
for _ in range(n_samples):
    z = [random.gauss(0.0, 1.0) for _ in range(3)]
    x = lower_matvec(L, z)
    c0.append(x[0])
    c1.append(x[1])
    c2.append(x[2])

print("Pair (0,1) target:", C[0][1], "sample corr:", stats.correlation(c0, c1))
print("Pair (0,2) target:", C[0][2], "sample corr:", stats.correlation(c0, c2))
print("Pair (1,2) target:", C[1][2], "sample corr:", stats.correlation(c1, c2))

Target correlation matrix validated: OK
Cholesky L (first row): [1.0, 0.0, 0.0]
Pair (0,1) target: 0.7 sample corr: 0.6990287019943572
Pair (0,2) target: 0.5 sample corr: 0.5033536075874913
Pair (1,2) target: 0.3 sample corr: 0.30464405373040254


In [6]:
from finstack.core.math.summation import kahan_sum, neumaier_sum


def naive_left_to_right(values):
    """Strict left-to-right float add (typical hand-written loop semantics)."""
    total = 0.0
    for v in values:
        total += v
    return total


# Large-magnitude cancellation: left-to-right IEEE addition can drop moderate terms.
values = [1e16] + [1.0] * 1000 + [-1e16]
builtin_sum = sum(values)
naive_cancel = naive_left_to_right(values)
kahan_cancel = kahan_sum(values)
neumaier_cancel = neumaier_sum(values)
print("Sum of [1e16, +1 (x1000), -1e16] — exact total should be 1000:")
print("  builtin sum():", builtin_sum)
print("  left-to-right:", naive_cancel)
print("  kahan:         ", kahan_cancel)
print("  neumaier:      ", neumaier_cancel)
print("  expected:      ", 1000.0)

Sum of [1e16, +1 (x1000), -1e16] — exact total should be 1000:
  builtin sum(): 1000.0
  left-to-right: 0.0
  kahan:          1000.0
  neumaier:       1000.0
  expected:       1000.0


## Takeaways

- **`finstack.core.math.linalg`** — Cholesky factorization and triangular solves for SPD systems; nested-matrix `validate_correlation_matrix` enforces standard correlation-matrix constraints.
- **`finstack.correlation`** — the same validation accepts **flat row-major** packed matrices `(values, n)`, which matches patterns in benchmarks and low-level pipelines.
- **`finstack.core.math.stats`** — means, variances, covariance, correlation, and quantiles for quick sanity checks on data and simulations.
- **Special functions** — Gaussian and gamma-related primitives for models that depend on \(\Phi\), \(\varphi\), or \(\ln\Gamma\).
- **Kahan / Neumaier** — use when summing long sequences where rounding error matters; Neumaier is often preferable when signs mix.
- **Correlated normals** — if \(Z\) is standard independent and \(C = LL^\top\), then \(LZ\) has correlation \(C\); sample correlations converge to targets as you increase paths.

**Next:** continue the curriculum notebook that builds on these numerics (simulation, curves, or instruments — depending on track 05).